# Model Comparison with GroupKFold

This notebook compares seven tabular models on the bottleneck severity dataset using the same preprocessing and feature engineering as the reference notebook, but with GroupKFold cross-validation and a cleaner, modular evaluation workflow.


In [21]:
import pickle
import warnings
from pathlib import Path
import platform

import joblib
import matplotlib
matplotlib.use('Agg')
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import seaborn as sns
from sklearn.ensemble import (
    ExtraTreesClassifier,
    HistGradientBoostingClassifier,
    HistGradientBoostingRegressor,
    RandomForestClassifier,
)
from sklearn.metrics import (
    accuracy_score,
    balanced_accuracy_score,
    classification_report,
    confusion_matrix,
    cohen_kappa_score,
    matthews_corrcoef,
    precision_recall_fscore_support,
)
from sklearn.model_selection import GroupKFold

try:
    from xgboost import XGBClassifier
except ImportError:
    XGBClassifier = None

try:
    from lightgbm import LGBMClassifier
except ImportError:
    LGBMClassifier = None

try:
    from catboost import CatBoostClassifier
except ImportError:
    CatBoostClassifier = None

warnings.filterwarnings('ignore')
pd.set_option('display.float_format', '{:.3f}'.format)
sns.set_theme(style='whitegrid')

RANDOM_STATE = 42
N_CLASSES = 4
# Use workspace absolute path as ROOT (Linux)
ROOT = Path('/home/anindta-ubuntu/FYP/PerformanceBottleneckDiagnosis').resolve()
INPUT_CSV = ROOT / 'src' / 'approach3' / 'perf_metrics.csv'
ARTIFACT_DIR = ROOT / 'src' / 'ml' / 'artifacts'
ARTIFACT_DIR.mkdir(parents=True, exist_ok=True)
TARGET_NAMES = ['Normal', 'Low', 'Medium', 'High']
TARGET_LABELS = [0, 1, 2, 3]
CLASS_NAMES_MAP = {0: 'Normal', 1: 'Low', 2: 'Medium', 3: 'High'}


## Configuration

Set the constants that control the notebook path, output artifacts, labels, and evaluation settings.

In [22]:
from pathlib import Path
import platform

import numpy as np
import pandas as pd

# Use workspace absolute path as ROOT (Linux)
ROOT = Path('/home/anindta-ubuntu/FYP/PerformanceBottleneckDiagnosis').resolve()
INPUT_CSV = ROOT / 'src' / 'approach3' / 'perf_metrics.csv'
ARTIFACT_DIR = ROOT / 'src' / 'ml' / 'artifacts'
ARTIFACT_DIR.mkdir(parents=True, exist_ok=True)

RANDOM_STATE = 42
N_CLASSES = 4
TARGET_NAMES = ['Normal', 'Low', 'Medium', 'High']
TARGET_LABELS = [0, 1, 2, 3]
CLASS_NAMES_MAP = {0: 'Normal', 1: 'Low', 2: 'Medium', 3: 'High'}


## Load Dataset

Read the performance metrics CSV and inspect the shape before cleaning.

In [23]:
df_raw = pd.read_csv(INPUT_CSV)
print(f'Shape: {df_raw.shape}')
print(df_raw.head(3))
print('\nSession labels:', df_raw['session_label'].nunique())

Shape: (526895, 44)
     timestamp_ns     pid  cpu             comm  ctx_switches  \
0  15253211214479   53292    4  TaskCon~ller #2             9   
1  15326325471859  158069    2           docker            10   
2  15200712630438  157330    6           docker             6   

   voluntary_switches  involuntary_switches  cpu_migrations  total_runtime_ns  \
0                   9                     0               1            256663   
1                   9                     1               1           3143651   
2                   6                     0               0           1642845   

   stall_ns  ...  syscall_error_count  mutex_contentions  avg_mutex_wait_ns  \
0     57817  ...                    0                  0                  0   
1     12296  ...                    0                  1                982   
2     49874  ...                    0                  0                  0   

   max_mutex_wait_ns  rwsem_read_contentions  avg_rwsem_read_wait_ns  \
0    

## Data Cleaning

Apply the same cleaning filters used in the reference notebook to avoid target leakage and keep the dataset consistent.

In [24]:
df = df_raw.copy()
before = len(df)

mask = (
    (df['pid'] > 0)
    & (df['ctx_switches'] > 0)
    & (df['latency_count'] > 0)
    & (df['total_runtime_ns'] > 0)
)
df = df.loc[mask].copy()

print(f'Rows removed: {before - len(df)}')
print(f'Rows retained: {len(df)}')

Rows removed: 95518
Rows retained: 431377


## Feature Engineering

Recreate the engineered features from the reference notebook without changing the feature set or leakage rules.

In [25]:
def _num(series: pd.Series) -> pd.Series:
    return pd.to_numeric(series, errors='coerce').fillna(0)


def _safe_div(numerator: pd.Series, denominator: pd.Series | int | float) -> pd.Series:
    den = denominator if isinstance(denominator, pd.Series) else pd.Series(denominator, index=numerator.index)
    den = pd.to_numeric(den, errors='coerce').replace(0, np.nan)
    return (pd.to_numeric(numerator, errors='coerce') / den).replace([np.inf, -np.inf], np.nan).fillna(0)


df = df.copy()
ctx = _num(df['ctx_switches'])
runtime = _num(df['total_runtime_ns'])
voluntary = _num(df['voluntary_switches'])
involuntary = _num(df['involuntary_switches'])
migrations = _num(df['cpu_migrations'])
minor_faults = _num(df['minor_faults'])
major_faults = _num(df['major_faults'])
kmalloc = _num(df['kmalloc_count'])
kfree = _num(df['kfree_count'])
alloc_bytes = _num(df['total_alloc_bytes'])
free_bytes = _num(df['total_free_bytes'])
large_pages = _num(df['large_page_allocs'])
syscalls = _num(df['syscall_count'])
reads = _num(df['read_count'])
writes = _num(df['write_count'])
read_bytes = _num(df['read_bytes'])
write_bytes = _num(df['write_bytes'])
io_bytes = read_bytes + write_bytes
futex = _num(df['futex_count'])
syscall_errors = _num(df['syscall_error_count'])
mutex_contentions = _num(df['mutex_contentions'])
avg_mutex_wait = _num(df['avg_mutex_wait_ns'])
rwsem_read = _num(df['rwsem_read_contentions'])
rwsem_write = _num(df['rwsem_write_contentions'])
rwsem_total = rwsem_read + rwsem_write

total_faults = minor_faults + major_faults

df['runtime_per_switch'] = _safe_div(runtime, ctx)
df['voluntary_ratio'] = _safe_div(voluntary, ctx)
df['involuntary_ratio'] = _safe_div(involuntary, ctx)
df['migration_ratio'] = _safe_div(migrations, ctx)
df['switches_per_runtime'] = _safe_div(ctx, runtime)
df['fault_rate'] = _safe_div(total_faults, runtime)
df['major_fault_ratio'] = _safe_div(major_faults, total_faults)
df['alloc_pressure'] = _safe_div(alloc_bytes, ctx)
df['free_pressure'] = _safe_div(free_bytes, ctx)
df['large_page_ratio'] = _safe_div(large_pages, kmalloc)
df['syscall_rate'] = _safe_div(syscalls, runtime)
df['bytes_per_syscall'] = _safe_div(io_bytes, syscalls)
df['futex_ratio'] = _safe_div(futex, syscalls)
df['io_intensity'] = _safe_div(reads + writes, syscalls)
df['syscall_error_ratio'] = _safe_div(syscall_errors, syscalls)
df['lock_pressure'] = mutex_contentions + rwsem_total
df['mutex_wait_per_contention'] = _safe_div(avg_mutex_wait * mutex_contentions, mutex_contentions)
df['rwsem_pressure'] = rwsem_total
df['rwsem_write_ratio'] = _safe_div(rwsem_write, rwsem_total)

cpu_pressure = df['switches_per_runtime'] + df['involuntary_ratio'] + df['migration_ratio']
memory_pressure = df['fault_rate'] + df['alloc_pressure'] + df['large_page_ratio']
io_pressure = df['syscall_rate'] + df['io_intensity'] + df['bytes_per_syscall']
lock_pressure = df['lock_pressure'] + df['mutex_wait_per_contention'] + df['rwsem_pressure']

df['cpu_memory_ratio'] = _safe_div(cpu_pressure, memory_pressure)
df['io_cpu_ratio'] = _safe_div(io_pressure, cpu_pressure)
df['lock_cpu_ratio'] = _safe_div(lock_pressure, cpu_pressure)
df['memory_lock_ratio'] = _safe_div(memory_pressure, lock_pressure)

log_cols = ['total_runtime_ns', 'avg_syscall_latency_ns', 'max_syscall_latency_ns', 'read_bytes', 'write_bytes', 'total_alloc_bytes']
for col in log_cols:
    if col in df.columns:
        df[f'log_{col}'] = np.log1p(df[col])

LEAKY_COLS = {
    'avg_stall_ns', 'stall_ns', 'max_stall_ns', 'latency_count',
    'stall_per_switch', 'log_stall_ns', 'log_avg_stall_ns', 'log_max_stall_ns',
    'stall_spike', 'log_stall_spike',
}

ALLOWED_FEATURES = [
    'ctx_switches', 'cpu_migrations', 'involuntary_switches', 'voluntary_switches',
    'involuntary_ratio', 'voluntary_ratio', 'avg_runq_ratio', 'runtime_per_switch',
    'minor_faults', 'major_faults', 'kmalloc_count', 'kfree_count',
    'total_alloc_bytes', 'alloc_pressure',
    'read_count', 'write_count', 'read_bytes', 'write_bytes', 'read_write_ratio', 'io_bytes_total',
    'mutex_contentions', 'avg_mutex_wait_ns', 'rwsem_read_contentions', 'rwsem_write_contentions', 'lock_pressure',
    'syscall_count', 'avg_syscall_latency_ns', 'futex_count',
    'log_total_runtime_ns', 'log_avg_syscall_latency_ns', 'log_max_syscall_latency_ns',
    'log_io_bytes_total', 'log_total_alloc_bytes',
]

FEATURE_COLS = [f for f in ALLOWED_FEATURES if f in df.columns]
assert not set(FEATURE_COLS) & LEAKY_COLS
print(f'Prepared {len(FEATURE_COLS)} features')

Prepared 30 features


## Target Encoding

Create ordinal bottleneck severity labels from the stall latency distribution using the same style of label generation as the project notebooks.

In [26]:
p50 = df['avg_stall_ns'].quantile(0.50)
p85 = df['avg_stall_ns'].quantile(0.85)
p97 = df['avg_stall_ns'].quantile(0.97)
bins = [0, p50, p85, p97, float('inf')]
labels = [0, 1, 2, 3]

df['y'] = pd.cut(df['avg_stall_ns'], bins=bins, labels=labels).astype(int)
df['y_name'] = df['y'].map(CLASS_NAMES_MAP)

print(f'P50={p50:.0f}, P85={p85:.0f}, P97={p97:.0f}')
print(df['y'].value_counts().sort_index().to_string())

P50=75551, P85=503952, P97=1964861
y
0    215690
1    150980
2     51765
3     12942


## GroupKFold Setup

Use group-aware cross-validation so the models are evaluated without leaking information between workloads or sessions.

In [27]:
X = df[FEATURE_COLS].fillna(0)
y = df['y'].astype(int)
groups = df['session_label'].astype(str)

splitter = GroupKFold(n_splits=5)
print(f'Using GroupKFold with {splitter.n_splits} folds and {groups.nunique()} groups')

Using GroupKFold with 5 folds and 50 groups


## Helper Evaluation Functions

Define reusable helpers for metrics, ordinal decoding, and confusion-matrix plotting so the comparison stays modular.

In [28]:
def compute_metrics(y_true, y_pred):
    precision, recall, f1, _ = precision_recall_fscore_support(y_true, y_pred, average='macro', zero_division=0)
    distance = np.abs(np.asarray(y_pred) - np.asarray(y_true))
    return {
        'accuracy': accuracy_score(y_true, y_pred),
        'macro_precision': float(precision),
        'macro_recall': float(recall),
        'macro_f1': float(f1),
        'balanced_accuracy': balanced_accuracy_score(y_true, y_pred),
        'mcc': matthews_corrcoef(y_true, y_pred),
        'quadratic_weighted_kappa': cohen_kappa_score(y_true, y_pred, weights='quadratic'),
        'mean_absolute_error': float(np.mean(distance)),
        'mean_squared_error': float(np.mean(distance ** 2)),
        'average_severity_distance': float(np.mean(distance)),
        'adjacent_accuracy': float(np.mean(distance <= 1)),
        'severe_error_rate': float(np.mean(distance >= 2)),
    }


def rounded_clipped_regression(pred):
    return np.clip(np.rint(np.asarray(pred)), 0, N_CLASSES - 1).astype(int)


def plot_confusion_matrix(y_true, y_pred, model_name):
    cm = confusion_matrix(y_true, y_pred, labels=TARGET_LABELS)
    fig, ax = plt.subplots(figsize=(6, 5))
    im = ax.imshow(cm, cmap='Blues')
    fig.colorbar(im, ax=ax, fraction=0.046, pad=0.04)
    ax.set_xticks(np.arange(len(TARGET_LABELS)))
    ax.set_xticklabels(TARGET_NAMES)
    ax.set_yticks(np.arange(len(TARGET_LABELS)))
    ax.set_yticklabels(TARGET_NAMES)
    ax.set_xlabel('Predicted class')
    ax.set_ylabel('Actual class')
    ax.set_title(f'{model_name} confusion matrix')
    for row in range(cm.shape[0]):
        for col in range(cm.shape[1]):
            ax.text(col, row, int(cm[row, col]), ha='center', va='center')
    fig.tight_layout()
    out_path = ARTIFACT_DIR / f'{model_name.lower().replace(" ", "_")}_confusion_matrix.png'
    fig.savefig(out_path, dpi=150)
    plt.close(fig)


def build_classification_report(y_true, y_pred, model_name):
    return classification_report(y_true, y_pred, labels=TARGET_LABELS, target_names=TARGET_NAMES, zero_division=0)


## Model Factory and Training

Create a generic factory that can instantiate each model with sensible defaults and train it fold by fold.

In [29]:
def make_model(name):
    if name == 'RandomForestClassifier':
        return RandomForestClassifier(n_estimators=200, class_weight='balanced', n_jobs=-1, random_state=RANDOM_STATE)
    if name == 'ExtraTreesClassifier':
        return ExtraTreesClassifier(n_estimators=200, class_weight='balanced', n_jobs=-1, random_state=RANDOM_STATE)
    if name == 'HistGradientBoostingClassifier':
        return HistGradientBoostingClassifier(random_state=RANDOM_STATE)
    if name == 'XGBoostClassifier':
        if XGBClassifier is None:
            raise ImportError('xgboost is not installed')
        return XGBClassifier(n_estimators=200, objective='multi:softprob', eval_metric='mlogloss', random_state=RANDOM_STATE, n_jobs=-1)
    if name == 'LightGBMClassifier':
        if LGBMClassifier is None:
            raise ImportError('lightgbm is not installed')
        return LGBMClassifier(n_estimators=200, random_state=RANDOM_STATE, n_jobs=-1)
    if name == 'CatBoostClassifier':
        if CatBoostClassifier is None:
            raise ImportError('catboost is not installed')
        return CatBoostClassifier(iterations=200, loss_function='MultiClass', silent=True, random_seed=RANDOM_STATE)
    if name == 'HistGradientBoostingRegressor':
        return HistGradientBoostingRegressor(random_state=RANDOM_STATE)
    raise ValueError(name)


def evaluate_model(name, X, y, groups):
    fold_rows = []
    fold_reports = []
    fold_predictions = []
    fold_confusion = []
    splitter = GroupKFold(n_splits=5)

    for fold, (train_idx, test_idx) in enumerate(splitter.split(X, y, groups=groups), start=1):
        X_train, X_test = X.iloc[train_idx], X.iloc[test_idx]
        y_train, y_test = y.iloc[train_idx], y.iloc[test_idx]

        model = make_model(name)
        if name == 'HistGradientBoostingRegressor':
            model.fit(X_train, y_train)
            pred = rounded_clipped_regression(model.predict(X_test))
        else:
            model.fit(X_train, y_train)
            pred = model.predict(X_test)

        metrics = compute_metrics(y_test, pred)
        metrics.update({'model': name, 'fold': fold})
        fold_rows.append(metrics)
        fold_reports.append((fold, build_classification_report(y_test, pred, name)))
        fold_confusion.append((fold, confusion_matrix(y_test, pred, labels=TARGET_LABELS)))
        fold_predictions.append(pd.DataFrame({'fold': fold, 'y_true': y_test.reset_index(drop=True), 'y_pred': pd.Series(pred).reset_index(drop=True)}))

    per_fold_df = pd.DataFrame(fold_rows)
    avg_metrics = per_fold_df[[col for col in per_fold_df.columns if col not in {'model', 'fold'}]].mean().to_dict()
    avg_metrics.update({'model': name})

    for fold, report in fold_reports:
        print(f'[{name}] Fold {fold}\n{report}\n')

    for fold, cm in fold_confusion:
        plot_confusion_matrix(y_test, pred, name)
        break

    return per_fold_df, avg_metrics, fold_predictions


## Run Model Comparison

Loop over the selected models, collect metrics, and save the comparison results for later analysis.

In [30]:
MODEL_NAMES = [
    'RandomForestClassifier',
    'ExtraTreesClassifier',
    'HistGradientBoostingClassifier',
    'XGBoostClassifier',
    'LightGBMClassifier',
    'CatBoostClassifier',
    'HistGradientBoostingRegressor',
]

ARTIFACT_DIR = ROOT / 'src' / 'ml' / 'artifacts'
ARTIFACT_DIR.mkdir(parents=True, exist_ok=True)

all_fold_rows = []
all_avg_rows = []
model_artifacts = {}

for model_name in MODEL_NAMES:
    print(f'\n=== Evaluating {model_name} ===')
    try:
        per_fold_df, avg_metrics, fold_predictions = evaluate_model(model_name, X, y, groups)
    except Exception as exc:
        print(f'[{model_name}] failed: {exc}')
        continue

    all_fold_rows.append(per_fold_df.assign(model=model_name))
    all_avg_rows.append(avg_metrics)

    if len(fold_predictions) > 0:
        preds_df = pd.concat(fold_predictions, ignore_index=True)
        preds_df.to_csv(ARTIFACT_DIR / f'{model_name.lower().replace(" ", "_")}_predictions.csv', index=False)
        model_artifacts[model_name] = {
            'model': make_model(model_name),
            'predictions': preds_df,
        }

if all_fold_rows:
    per_fold_metrics = pd.concat(all_fold_rows, ignore_index=True)
    per_fold_metrics.to_csv(ARTIFACT_DIR / 'per_fold_metrics.csv', index=False)

if all_avg_rows:
    comparison_metrics = pd.DataFrame(all_avg_rows)
    comparison_metrics.to_csv(ARTIFACT_DIR / 'comparison_metrics.csv', index=False)

comparison_metrics.head()


=== Evaluating RandomForestClassifier ===
[RandomForestClassifier] Fold 1
              precision    recall  f1-score   support

      Normal       0.66      0.63      0.65     28775
         Low       0.70      0.71      0.70     40125
      Medium       0.36      0.42      0.39     12398
        High       0.68      0.54      0.60      4621

    accuracy                           0.63     85919
   macro avg       0.60      0.58      0.59     85919
weighted avg       0.64      0.63      0.63     85919


[RandomForestClassifier] Fold 2
              precision    recall  f1-score   support

      Normal       0.73      0.65      0.69     36865
         Low       0.67      0.68      0.68     36981
      Medium       0.31      0.42      0.36      9872
        High       0.66      0.76      0.71      2511

    accuracy                           0.64     86229
   macro avg       0.59      0.63      0.61     86229
weighted avg       0.66      0.64      0.65     86229


[RandomForestClassifi

,accuracy,macro_precision,macro_recall,macro_f1,balanced_accuracy,mcc,quadratic_weighted_kappa,mean_absolute_error,mean_squared_error,average_severity_distance,adjacent_accuracy,severe_error_rate,model
0,0.609,0.531,0.555,0.531,0.555,0.379,0.412,0.526,0.817,0.526,0.876,0.124,RandomForestClassifier
1,0.607,0.534,0.526,0.522,0.526,0.356,0.396,0.518,0.785,0.518,0.884,0.116,ExtraTreesClassifier
2,0.694,0.633,0.570,0.591,0.570,0.458,0.514,0.383,0.550,0.383,0.929,0.071,HistGradientBoostingClassifier
3,0.671,0.626,0.566,0.585,0.566,0.438,0.469,0.432,0.650,0.432,0.903,0.097,XGBoostClassifier
4,0.702,0.645,0.572,0.596,0.572,0.471,0.534,0.370,0.525,0.370,0.935,0.065,LightGBMClassifier


## Rankings and Export

Rank the evaluated models by the requested metrics and export the selected model for reuse.

In [31]:
if 'comparison_metrics' in globals() and not comparison_metrics.empty:
    ranking_cols = ['macro_f1', 'accuracy', 'balanced_accuracy', 'quadratic_weighted_kappa', 'adjacent_accuracy']
    ranking_cols = [c for c in ranking_cols if c in comparison_metrics.columns]
    comparison_metrics = comparison_metrics.sort_values(by=ranking_cols, ascending=False).reset_index(drop=True)
    comparison_metrics.to_csv(ARTIFACT_DIR / 'comparison_metrics.csv', index=False)

    best_model_name = comparison_metrics.iloc[0]['model']
    best_model = make_model(best_model_name)
    best_model.fit(X, y)
    with open(ARTIFACT_DIR / 'best_model.pkl', 'wb') as fh:
        pickle.dump(best_model, fh)
    print(f'Best model saved: {best_model_name}')

comparison_metrics.head()

[LightGBM] [Info] Auto-choosing col-wise multi-threading, the overhead of testing was 0.038959 seconds.
You can set `force_col_wise=true` to remove the overhead.
[LightGBM] [Info] Total Bins 7140
[LightGBM] [Info] Number of data points in the train set: 431377, number of used features: 28
[LightGBM] [Info] Start training from score -0.693140
[LightGBM] [Info] Start training from score -1.049835
[LightGBM] [Info] Start training from score -2.120268
[LightGBM] [Info] Start training from score -3.506505
Best model saved: LightGBMClassifier


,accuracy,macro_precision,macro_recall,macro_f1,balanced_accuracy,mcc,quadratic_weighted_kappa,mean_absolute_error,mean_squared_error,average_severity_distance,adjacent_accuracy,severe_error_rate,model
0,0.702,0.645,0.572,0.596,0.572,0.471,0.534,0.370,0.525,0.370,0.935,0.065,LightGBMClassifier
1,0.694,0.633,0.570,0.591,0.570,0.458,0.514,0.383,0.550,0.383,0.929,0.071,HistGradientBoostingClassifier
2,0.671,0.626,0.566,0.585,0.566,0.438,0.469,0.432,0.650,0.432,0.903,0.097,XGBoostClassifier
3,0.609,0.531,0.555,0.531,0.555,0.379,0.412,0.526,0.817,0.526,0.876,0.124,RandomForestClassifier
4,0.607,0.534,0.526,0.522,0.526,0.356,0.396,0.518,0.785,0.518,0.884,0.116,ExtraTreesClassifier


Note: Some of the relative paths might not work since i moved the artifacts around